# 可选实验：多变量线性回归

在本实验中，你将扩展数据结构和此前开发的例程，以支持多个特征。有几个例程做了更新，使实验看起来较长，但相对于此前的例程只是做了小幅调整，因此复习起来很快。
# 目录
- [&nbsp;&nbsp;1.1 目标](#toc_15456_1.1)
- [&nbsp;&nbsp;1.2 工具](#toc_15456_1.2)
- [&nbsp;&nbsp;1.3 记号](#toc_15456_1.3)
- [2 问题陈述](#toc_15456_2)
- [&nbsp;&nbsp;2.1 包含样本的矩阵 X](#toc_15456_2.1)
- [&nbsp;&nbsp;2.2 参数向量 w、b](#toc_15456_2.2)
- [3 多变量的模型预测](#toc_15456_3)
- [&nbsp;&nbsp;3.1 单次预测：逐元素](#toc_15456_3.1)
- [&nbsp;&nbsp;3.2 单次预测：向量形式](#toc_15456_3.2)
- [4 多变量代价计算](#toc_15456_4)
- [5 多变量梯度下降](#toc_15456_5)
- [&nbsp;&nbsp;5.1 多变量梯度计算](#toc_15456_5.1)
- [&nbsp;&nbsp;5.2 多变量梯度下降](#toc_15456_5.2)
- [6 恭喜](#toc_15456_6)


<a name="toc_15456_1.1"></a>
## 1.1 目标
- 扩展我们的回归模型例程以支持多个特征
    - 扩展数据结构以支持多个特征
    - 重写预测、代价和梯度例程以支持多个特征
    - 利用 NumPy `np.dot` 将它们的实现向量化，以提升速度并简化代码

<a name="toc_15456_1.2"></a>
## 1.2 工具
在本实验中，我们将使用：
- NumPy，一个流行的科学计算库
- Matplotlib，一个流行的数据绘图库

In [ ]:
import copy, math
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
np.set_printoptions(precision=2)  # 降低 numpy 数组的显示精度

<a name="toc_15456_1.3"></a>
## 1.3 记号
下面是你将会遇到的一些记号的汇总，已针对多特征做了更新。

| 一般 记号 | 说明 | Python（如适用） |
|:---|:---|:---|
| $a$ | 标量，非粗体 |  |
| $\mathbf{a}$ | 向量，粗体 |  |
| $\mathbf{A}$ | 矩阵，粗体大写 |  |
| **回归** |  |  |
| $\mathbf{X}$ | 训练样本矩阵 | `X_train` |
| $\mathbf{y}$ | 训练样本目标值 | `y_train` |
| $\mathbf{x}^{(i)}$, $y^{(i)}$ | 第 $i$ 个训练样本 | `X[i]`, `y[i]` |
| m | 训练样本的数量 | `m` |
| n | 每个样本中的特征数量 | `n` |
| $\mathbf{w}$ | 参数：权重 | `w` |
| $b$ | 参数：偏置 | `b` |
| $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ | 以 $\mathbf{w},b$ 为参数、在 $\mathbf{x^{(i)}}$ 处求得的模型评估结果：$f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)}+b$ | `f_wb` |


<a name="toc_15456_2"></a>
# 2 问题陈述

你将使用房价预测这个引入性例子。训练数据集包含三个样本、四个特征（面积、卧室数、楼层数和房龄），如下表所示。注意，与之前的实验不同，这里的面积单位是平方英尺而不是 1000 平方英尺。这会带来一个问题，你将在下一个实验中解决！

| 面积（平方英尺） | 卧室数  | 楼层数 | 房龄 | 价格（千美元）  |   
| ----------------| ------------------- |----------------- |--------------|-------------- |  
| 2104            | 5                   | 1                | 45           | 460           |  
| 1416            | 3                   | 2                | 40           | 232           |  
| 852             | 2                   | 1                | 35           | 178           |  

你将用这些数值建立一个线性回归模型，从而可以预测其他住房的价格。例如，一套 1200 平方英尺、3 间卧室、1 层、房龄 40 年的住房。

请运行下面的代码单元，创建 `X_train` 和 `y_train` 变量。

In [ ]:
X_train = np.array([[2104, 5, 1, 45], [1416, 3, 2, 40], [852, 2, 1, 35]])
y_train = np.array([460, 232, 178])

<a name="toc_15456_2.1"></a>
## 2.1 包含样本的矩阵 X
与上表类似，样本存储在 NumPy 矩阵 `X_train` 中。矩阵的每一行表示一个样本。当你有 $m$ 个训练样本（本例中 $m$ 为三）且有 $n$ 个特征（本例中为四）时，$\mathbf{X}$ 是一个维度为 ($m$, $n$) 的矩阵（m 行 n 列）。


$$\mathbf{X} = 
\begin{pmatrix}
 x^{(0)}_0 & x^{(0)}_1 & \cdots & x^{(0)}_{n-1} \\ 
 x^{(1)}_0 & x^{(1)}_1 & \cdots & x^{(1)}_{n-1} \\
 \cdots \\
 x^{(m-1)}_0 & x^{(m-1)}_1 & \cdots & x^{(m-1)}_{n-1} 
\end{pmatrix}
$$
记号：
- $\mathbf{x}^{(i)}$ 是包含样本 i 的向量。$\mathbf{x}^{(i)}$ $ = (x^{(i)}_0, x^{(i)}_1, \cdots,x^{(i)}_{n-1})$
- $x^{(i)}_j$ 是样本 i 中的第 j 个元素。圆括号中的上标表示样本编号，下标表示元素。

显示输入数据。

In [ ]:
# 数据存放在 numpy 数组/矩阵中
print(f"X Shape: {X_train.shape}, X Type:{type(X_train)})")
print(X_train)
print(f"y Shape: {y_train.shape}, y Type:{type(y_train)})")
print(y_train)

<a name="toc_15456_2.2"></a>
## 2.2 参数向量 w、b

* $\mathbf{w}$ 是含 $n$ 个元素的向量。
  - 每个元素包含与一个特征相关的参数。
  - 在我们的数据集中，n 为 4。
  - 在概念上，我们把它画成列向量

$$\mathbf{w} = \begin{pmatrix}
w_0 \\ 
w_1 \\
\cdots\\
w_{n-1}
\end{pmatrix}
$$
* $b$ 是一个标量参数。  

为便于演示，$\mathbf{w}$ 和 $b$ 会被载入一些接近最优的初始选定值。$\mathbf{w}$ 是一个一维 NumPy 向量。

In [ ]:
b_init = 785.1811367994083
w_init = np.array([ 0.39133535, 18.75376741, -53.36032453, -26.42131618])
print(f"w_init shape: {w_init.shape}, b_init type: {type(b_init)}")

<a name="toc_15456_3"></a>
# 3 多变量的模型预测
多变量下模型的预测由线性模型给出：

$$ f_{\mathbf{w},b}(\mathbf{x}) =  w_0x_0 + w_1x_1 +... + w_{n-1}x_{n-1} + b \tag{1}$$
或者用向量记号：
$$ f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w} \cdot \mathbf{x} + b  \tag{2} $$ 
其中 $\cdot$ 表示向量`点积`

为演示点积，我们将分别用 (1) 和 (2) 实现预测。

<a name="toc_15456_3.1"></a>
## 3.1 单次预测：逐元素
我们此前的预测是把一个特征值乘以一个参数，再加上偏置参数。把此前的预测实现直接扩展到多特征，就是用循环遍历每个元素来实现上面的 (1)，把每个元素与其参数相乘，最后再加上偏置参数。


In [ ]:
def predict_single_loop(x, w, b): 
    """使用线性回归做单样本预测
   
    参数：
      x (ndarray): 形状 (n,)，含多个特征的样本
      w (ndarray): 形状 (n,)，模型参数
      b (scalar):  模型参数
     
    返回：
      p (scalar):  预测值"""
    n = x.shape[0]
    p = 0
    for i in range(n):
        p_i = x[i] * w[i]  
        p = p + p_i         
    p = p + b                
    return p

In [ ]:
# 从训练数据中取一行
x_vec = X_train[0,:]
print(f"x_vec shape {x_vec.shape}, x_vec value: {x_vec}")

# 做一次预测
f_wb = predict_single_loop(x_vec, w_init, b_init)
print(f"f_wb shape {f_wb.shape}, prediction: {f_wb}")

注意 `x_vec` 的形状。它是含 4 个元素的一维 NumPy 向量，(4,)。结果 `f_wb` 是一个标量。

<a name="toc_15456_3.2"></a>
## 3.2 单次预测：向量形式

注意上面的公式 (1) 可以用 (2) 那样的点积来实现。我们可以利用向量运算来加速预测。

回忆 Python/NumPy 实验中的内容：NumPy 的 `np.dot()`[[链接](https://numpy.org/doc/stable/reference/generated/numpy.dot.html)] 可用于执行向量点积。

In [ ]:
def predict(x, w, b): 
    """使用线性回归做单样本预测
    参数：
      x (ndarray): 形状 (n,)，含多个特征的样本
      w (ndarray): 形状 (n,)，模型参数
      b (scalar):             模型参数
     
    返回：
      p (scalar):  预测值"""
    p = np.dot(x, w) + b     
    return p    

In [ ]:
# 从训练数据中取一行
x_vec = X_train[0,:]
print(f"x_vec shape {x_vec.shape}, x_vec value: {x_vec}")

# 做一次预测
f_wb = predict(x_vec,w_init, b_init)
print(f"f_wb shape {f_wb.shape}, prediction: {f_wb}")

结果和形状都与之前用循环的版本相同。今后，这类运算都将使用 `np.dot`。现在预测只是一条语句。大多数例程会直接实现它，而不是调用单独的 predict 例程。

<a name="toc_15456_4"></a>
# 4 多变量代价计算
多变量代价函数 $J(\mathbf{w},b)$ 的公式为：
$$J(\mathbf{w},b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})^2 \tag{3}$$ 
其中：
$$ f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)} + b  \tag{4} $$ 


与之前的实验不同，这里的 $\mathbf{w}$ 和 $\mathbf{x}^{(i)}$ 是向量而非标量，以支持多个特征。

下面是公式 (3) 和 (4) 的一种实现。注意它使用了*本课程的标准写法*，即对所有 `m` 个样本做一次 for 循环。

In [ ]:
def compute_cost(X, y, w, b): 
    """计算代价
    参数：
      X (ndarray (m,n)): 数据，m 个样本，n 个特征
      y (ndarray (m,)) : 目标值
      w (ndarray (n,)) : 模型参数
      b (scalar)       : 模型参数
     
    返回：
      cost (scalar): 代价"""
    m = X.shape[0]
    cost = 0.0
    for i in range(m):                                
        f_wb_i = np.dot(X[i], w) + b           #(n,)(n,) = 标量（见 np.dot）
        cost = cost + (f_wb_i - y[i])**2       #标量
    cost = cost / (2 * m)                      #标量
    return cost

In [ ]:
# 用我们预先选好的最优参数计算并显示代价。
cost = compute_cost(X_train, y_train, w_init, b_init)
print(f'Cost at optimal w : {cost}')

**预期结果**：最优 w 处的代价：1.5578904045996674e-12

<a name="toc_15456_5"></a>
# 5 多变量梯度下降
多变量的梯度下降：

$$\begin{align*} \text{repeat}&\text{ until convergence:} \; \lbrace \newline\;
& w_j = w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{5}  \; & \text{for j = 0..n-1}\newline
&b\ \ = b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}  \newline \rbrace
\end{align*}$$

其中，n 是特征数量，参数 $w_j$、$b$ 同时更新，且

$$
\begin{align}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \tag{6}  \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{7}
\end{align}
$$
* m 是数据集中训练样本的数量

    
*  $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ 是模型的预测值，而 $y^{(i)}$ 是目标值

<a name="toc_15456_5.1"></a>
## 5.1 多变量梯度计算
下面是计算 (6) 和 (7) 的一种实现。实现方式有很多。在这个版本中，有一个
- 遍历所有 m 个样本的外层循环。
    - 该样本的 $\frac{\partial J(\mathbf{w},b)}{\partial b}$ 可以直接计算并累加
    - 在遍历所有 n 个特征的第二个循环中：
        - 对每个 $w_j$ 计算 $\frac{\partial J(\mathbf{w},b)}{\partial w_j}$。
   

In [ ]:
def compute_gradient(X, y, w, b): 
    """计算线性回归的梯度
    参数：
      X (ndarray (m,n)): 数据，m 个样本，n 个特征
      y (ndarray (m,)) : 目标值
      w (ndarray (n,)) : 模型参数
      b (scalar)       : 模型参数
     
    返回：
      dj_dw (ndarray (n,)): 代价对参数 w 的梯度。
      dj_db (scalar):       代价对参数 b 的梯度。"""
    m,n = X.shape           #（样本数，特征数）
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range(m):                             
        err = (np.dot(X[i], w) + b) - y[i]   
        for j in range(n):                         
            dj_dw[j] = dj_dw[j] + err * X[i, j]    
        dj_db = dj_db + err                        
    dj_dw = dj_dw / m                                
    dj_db = dj_db / m                                
        
    return dj_db, dj_dw

In [ ]:
#计算并显示梯度
tmp_dj_db, tmp_dj_dw = compute_gradient(X_train, y_train, w_init, b_init)
print(f'dj_db at initial w,b: {tmp_dj_db}')
print(f'dj_dw at initial w,b: \n {tmp_dj_dw}')

**预期结果**：
初始 w,b 处的 dj_db：-1.6739251122999121e-06
初始 w,b 处的 dj_dw：
 [-2.73e-03 -6.27e-06 -2.22e-06 -6.92e-05]  

<a name="toc_15456_5.2"></a>
## 5.2 多变量梯度下降
下面的例程实现上面的公式 (5)。

In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters): 
    """执行批量梯度下降以学习 theta。以学习率 alpha 走
    num_iters 个梯度步来更新 theta
   
    参数：
      X (ndarray (m,n))   : 数据，m 个样本，n 个特征
      y (ndarray (m,))    : 目标值
      w_in (ndarray (n,)) : 模型参数的初始值
      b_in (scalar)       : 模型参数的初始值
      cost_function       : 用于计算代价的函数
      gradient_function   : 用于计算梯度的函数
      alpha (float)       : 学习率
      num_iters (int)     : 运行梯度下降的迭代次数
     
    返回：
      w (ndarray (n,)) : 更新后的参数值
      b (scalar)       : 更新后的参数值"""
    
    # 用于存放代价 J 以及每次迭代的 w，主要供后续绘图使用
    J_history = []
    w = copy.deepcopy(w_in)  # 避免在函数内修改全局变量 w
    b = b_in
    
    for i in range(num_iters):

        # 计算梯度并更新参数
        dj_db,dj_dw = gradient_function(X, y, w, b)   ##None

        # 用 w、b、alpha 和梯度更新参数
        w = w - alpha * dj_dw               ##None
        b = b - alpha * dj_db               ##None
      
        # 每次迭代保存代价 J
        if i<100000:      # 防止资源耗尽
            J_history.append( cost_function(X, y, w, b))

        # 每 10 次迭代打印一次代价；若迭代次数少于 10，则打印所有迭代
        if i% math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]:8.2f}   ")
        
    return w, b, J_history #返回最终的 w、b 以及用于绘图的 J 历史

在下一个单元格中你将测试该实现。

In [ ]:
# 初始化参数
initial_w = np.zeros_like(w_init)
initial_b = 0.
# 一些梯度下降设置
iterations = 1000
alpha = 5.0e-7
# 运行梯度下降
w_final, b_final, J_hist = gradient_descent(X_train, y_train, initial_w, initial_b,
                                                    compute_cost, compute_gradient, 
                                                    alpha, iterations)
print(f"b,w found by gradient descent: {b_final:0.2f},{w_final} ")
m,_ = X_train.shape
for i in range(m):
    print(f"prediction: {np.dot(X_train[i], w_final) + b_final:0.2f}, target value: {y_train[i]}")

**预期结果**：
由梯度下降得到的 b,w：-0.00,[ 0.2   0.   -0.01 -0.07]   
预测值：426.19，目标值：460  
预测值：286.17，目标值：232  
预测值：171.47，目标值：178  

In [ ]:
# 绘制代价随迭代次数的变化
fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout=True, figsize=(12, 4))
ax1.plot(J_hist)
ax2.plot(100 + np.arange(len(J_hist[100:])), J_hist[100:])
ax1.set_title("Cost vs. iteration");  ax2.set_title("Cost vs. iteration (tail)")
ax1.set_ylabel('Cost')             ;  ax2.set_ylabel('Cost') 
ax1.set_xlabel('iteration step')   ;  ax2.set_xlabel('iteration step') 
plt.show()

*这些结果并不令人满意*！代价仍在下降，我们的预测还不够准确。下一个实验将探讨如何改进。


<a name="toc_15456_6"></a>
# 6 恭喜！
在本实验中你：
- 重新开发了线性回归的例程，现在支持多个变量。
- 利用 NumPy `np.dot` 将实现向量化